In [ ]:
# =====================
# 0) CONFIG / CONSTANTS
# =====================
"""
Purpose:
  Centralize the inputs and settings for the notebook so you only edit one place.

Notes:
  - Use absolute paths for local files.
  - Keep table identifiers here to avoid typos later.
"""

from pathlib import Path
from datetime import datetime

# ---- Run identifier (one timestamp used everywhere in this notebook run)
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- Target Snowflake objects
TARGET_DB = "EDLDB"
TARGET_SCHEMA = "SC_SANDBOX"
TARGET_TABLE = "VC_CPFR_VENDOR_EMAIL"
TARGET_FQN = f"{TARGET_DB}.{TARGET_SCHEMA}.{TARGET_TABLE}"

# ---- Staging location (your personal namespace)
STAGE_DB = "USER$NMILES1"
STAGE_SCHEMA = "WORK"    # or "SCRATCH"
STAGE_TABLE = f"VC_CPFR_VENDOR_EMAIL_STAGE_{RUN_TS}"
STAGE_FQN = f"{STAGE_DB}.{STAGE_SCHEMA}.{STAGE_TABLE}"

# ---- Local file inputs/outputs
CSV_PATH = Path("/Users/nmiles1/Library/CloudStorage/OneDrive-Chewy.com,LLC/Documents/cpfr_local/ad_hoc_reqs/CPFR_Vendor_Completion_Upload/Unapplied_Vendors.csv")

OUT_DIR = Path.home() / "Documents" / "cpfr_local" / "ad_hoc_reqs" / "CPFR_Vendor_Completion_Upload"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BACKUP_PATH = OUT_DIR / f"{TARGET_TABLE}_backup_{RUN_TS}.csv"
CONFLICTS_PATH = OUT_DIR / f"{TARGET_TABLE}_conflicts_{RUN_TS}.csv"  # optional but handy

# ---- Primary key concept for validation
KEY_COL = "Vendor Number"

print("Target table :", TARGET_FQN)
print("Stage table  :", STAGE_FQN)
print("CSV path     :", CSV_PATH)
print("Output folder:", OUT_DIR.resolve())
print("Backup path  :", BACKUP_PATH.resolve())
print("Target FQN:", TARGET_FQN)
print("Stage  FQN:", STAGE_FQN)

In [ ]:
# =======================
# 1) GET ACTIVE SESSION (Snowflake Notebook)
# =======================
"""
Purpose:
  In Snowflake Workspaces/Notebooks, reuse the already-authenticated session.

Notes:
  - No snowflake.connector needed.
  - Role/warehouse are whatever your notebook session is currently using.
  - You can still run USE ROLE / USE WAREHOUSE via session.sql(...).
"""

import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

sf_execute("USE ROLE SC_USER")
sf_execute("USE DATABASE EDLDB")
sf_execute("USE SCHEMA SC_SANDBOX")
# If needed for write operations:
# sf_execute("USE WAREHOUSE <YOUR_WAREHOUSE>")
sf_query_df("SELECT CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE()")

def sf_execute(sql: str):
    """Execute a SQL statement (DDL/DML) using the active Snowpark session."""
    session.sql(sql).collect()

def sf_query_df(sql: str) -> pd.DataFrame:
    """Run a SELECT and return a pandas DataFrame."""
    return session.sql(sql).to_pandas()

sf_query_df("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_WAREHOUSE()")

In [ ]:
from pathlib import Path

CSV_PATH = Path.cwd() / "cpfr_uploads" / "Unapplied_Vendors.csv"
CSV_PATH.exists(), str(CSV_PATH)

In [ ]:
# =====================
# 2) READ + INSPECT CSV
# =====================
"""
Purpose:
  Load the CSV and do basic visual inspection.

Notes:
  - We read everything as string first to avoid dtype surprises.
  - We'll normalize column names next.
"""

def read_csv_as_strings(path: Path) -> pd.DataFrame:
    """
    Read CSV into pandas with all columns as strings.

    Parameters:
      path: Path to CSV file

    Returns:
      DataFrame with string columns (NaNs possible for empty cells).
    """
    df = pd.read_csv(path, dtype=str, keep_default_na=True)
    return df

df_csv_raw = read_csv_as_strings(CSV_PATH)

print("CSV shape:", df_csv_raw.shape)
df_csv_raw.head(10)

In [ ]:
# ========================
# 3) CLEAN / NORMALIZE CSV
# ========================
"""
Purpose:
  Make the data consistent:
  - Strip whitespace in headers and string fields
  - Standardize missing values

Notes:
  - We do NOT change business content beyond trimming.
  - Email list normalization (splitting/sorting) can be added if desired later.
"""

def normalize_headers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize column headers by stripping whitespace.

    Returns a copy of the DataFrame.
    """
    df2 = df.copy()
    df2.columns = [c.strip() for c in df2.columns]
    return df2

def strip_string_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strip leading/trailing whitespace from all object/string columns.
    Leaves NaNs as NaNs.
    """
    df2 = df.copy()
    for col in df2.columns:
        df2[col] = df2[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    return df2

df_csv = normalize_headers(df_csv_raw)
df_csv = strip_string_fields(df_csv)

print("Columns:", list(df_csv.columns))
df_csv.head(10)

In [ ]:
# ======================================
# 4) VALIDATE: KEY COLUMN + DUPES IN CSV
# ======================================
"""
Purpose:
  Ensure Vendor Number exists and is unique in the CSV.

Outputs:
  - A duplicates DataFrame (if any)
  - A per-vendor diff report showing which columns vary
"""

def find_key_duplicates(df: pd.DataFrame, key_col: str) -> pd.DataFrame:
    """
    Return all rows that belong to duplicated key groups.

    If no duplicates, returns an empty DataFrame.
    """
    mask = df.duplicated(subset=[key_col], keep=False)
    return df.loc[mask].sort_values(key_col)

def columns_that_vary_within_group(group: pd.DataFrame) -> list:
    """
    For a group of rows (same Vendor Number), return list of columns whose values differ.
    """
    varying = []
    for col in group.columns:
        # nunique(dropna=False) counts NaN as a distinct value
        if group[col].nunique(dropna=False) > 1:
            varying.append(col)
    return varying

def summarize_duplicate_differences(df_dupes: pd.DataFrame, key_col: str) -> pd.DataFrame:
    """
    Produce a summary table: for each Vendor Number with duplicates,
    list which columns differ across rows.
    """
    if df_dupes.empty:
        return pd.DataFrame(columns=[key_col, "varying_columns"])

    rows = []
    for k, grp in df_dupes.groupby(key_col):
        rows.append({key_col: k, "varying_columns": ", ".join(columns_that_vary_within_group(grp))})
    return pd.DataFrame(rows).sort_values(key_col)

# ---- Basic key presence and null check
assert KEY_COL in df_csv.columns, f"CSV is missing required key column: {KEY_COL}"

null_keys = df_csv[df_csv[KEY_COL].isna() | (df_csv[KEY_COL].astype(str).str.strip() == "")]
print("Null/blank Vendor Number rows:", len(null_keys))
null_keys.head(10)

In [ ]:
df_dupes_csv = find_key_duplicates(df_csv, KEY_COL)
print("Duplicate Vendor Number rows in CSV:", len(df_dupes_csv))

dup_summary_csv = summarize_duplicate_differences(df_dupes_csv, KEY_COL)
dup_summary_csv.head(20)

In [ ]:
# If you want to inspect one duplicate Vendor Number visually:
if not df_dupes_csv.empty:
    example_vendor = df_dupes_csv[KEY_COL].iloc[0]
    print("Example duplicate Vendor Number:", example_vendor)
    df_dupes_csv[df_dupes_csv[KEY_COL] == example_vendor].T

In [ ]:
# ============================================
# 5) OPTIONAL: CHECK DUPES AGAINST EXISTING DB
# ============================================
"""
Purpose:
  Prevent inserting Vendor Numbers that already exist in the DB (if desired).

Notes:
  - We fetch just Vendor Number from the target table for fast comparison.
"""

df_db_keys = sf_query_df(f'SELECT "{KEY_COL}" AS KEY FROM {TARGET_FQN}')
print("DB keys shape:", df_db_keys.shape)
df_db_keys.head()

In [ ]:
csv_keys = set(df_csv[KEY_COL].dropna().astype(str))
db_keys  = set(df_db_keys["KEY"].dropna().astype(str))

already_in_db = sorted(csv_keys.intersection(db_keys))
print("Vendor Numbers in CSV that already exist in DB:", len(already_in_db))

pd.Series(already_in_db).head(20)

In [ ]:
# Create a "clean insert set" by excluding existing Vendor Numbers (adjust if you prefer MERGE behavior)
df_to_insert = df_csv[~df_csv[KEY_COL].astype(str).isin(already_in_db)].copy()

print("Rows to insert after excluding existing keys:", len(df_to_insert))
df_to_insert.head(10)

In [ ]:
# =========================
# 6) STAGE DATA IN SNOWFLAKE (TEMP TABLE)
# =========================
"""
Purpose:
  Stage the cleaned CSV rows in a session-scoped temp table.
Why:
  - Avoids restrictions on personal databases (USER$...).
  - Avoids cluttering shared schemas (temp table is not persistent).
  - Still allows inspection during this session.
"""

# Ensure you're in the right role/db/schema
sf_execute("USE ROLE SC_USER")
sf_execute("USE DATABASE EDLDB")
sf_execute("USE SCHEMA SC_SANDBOX")

TEMP_STAGE_NAME = f"TMP_VC_CPFR_VENDOR_EMAIL_STAGE_{RUN_TS}"

# Write as a TEMP table
sp_stage = session.create_dataframe(df_to_insert)
sp_stage.write.mode("overwrite").save_as_table(TEMP_STAGE_NAME, table_type="temp")

# Inspect
sf_query_df(f"SELECT COUNT(*) AS CNT FROM {TEMP_STAGE_NAME}")
sf_query_df(f"SELECT * FROM {TEMP_STAGE_NAME} LIMIT 10")

In [ ]:
# ===========================================
# 7) INSERT INTO TARGET (INSERT-ONLY VERSION)
# ===========================================
"""
Purpose:
  Insert staged rows into the target table.

Notes:
  - This assumes df_to_insert matches the target columns.
  - If columns differ, we should explicitly map them.
"""

df_target_cols = sf_query_df(f"""
SELECT COLUMN_NAME
FROM {TARGET_DB}.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = '{TARGET_SCHEMA}'
  AND TABLE_NAME = '{TARGET_TABLE}'
ORDER BY ORDINAL_POSITION
""")

target_cols = df_target_cols["COLUMN_NAME"].tolist()
common_cols = [c for c in df_to_insert.columns if c in target_cols]
missing_in_csv = [c for c in target_cols if c not in df_to_insert.columns]

print("Common cols:", common_cols)
print("Target cols missing in CSV (will be NULL/default):", missing_in_csv)

before_cnt = sf_query_df(f"SELECT COUNT(*) AS CNT FROM {TARGET_FQN}")["CNT"].iloc[0]
stage_cnt  = sf_query_df(f"SELECT COUNT(*) AS CNT FROM {TEMP_STAGE_NAME}")["CNT"].iloc[0]

skipped_cnt = sf_query_df(f"""
SELECT COUNT(*) AS CNT
FROM {TEMP_STAGE_NAME} s
WHERE EXISTS (
  SELECT 1
  FROM {TARGET_FQN} t
  WHERE TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
)
""")["CNT"].iloc[0]

print("Before CNT:", before_cnt)
print("Stage  CNT:", stage_cnt)
print("Would be skipped:", skipped_cnt)
print("Would be inserted:", stage_cnt - skipped_cnt)

col_list_sql = ", ".join([f'"{c}"' for c in common_cols])

sf_execute(f"""
INSERT INTO {TARGET_FQN} ({col_list_sql})
SELECT {col_list_sql}
FROM {TEMP_STAGE_NAME} s
WHERE NOT EXISTS (
  SELECT 1
  FROM {TARGET_FQN} t
  WHERE TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
)
""")

after_cnt = sf_query_df(f"SELECT COUNT(*) AS CNT FROM {TARGET_FQN}")["CNT"].iloc[0]
print("After CNT :", after_cnt)
print("Inserted? :", after_cnt - before_cnt)

## After-action validation that there are no Vendor Number duplicates in the target:

sf_query_df(f"""
SELECT "{KEY_COL}", COUNT(*) AS CNT
FROM {TARGET_FQN}
GROUP BY 1
HAVING COUNT(*) > 1
ORDER BY CNT DESC
""").head(50)

In [ ]:
# ==============================
# 8) EXPORT BACKUP TO USER STAGE + COPY INTO WORKSPACE FILES (robust)
# ==============================
"""
Purpose:
  1) Confirm the backup CSV exists in your user stage (@~) via LIST.
  2) Read the staged CSV back as raw lines.
  3) Write a copy into the Workspaces filesystem so it appears in the left file tree
     (and can be downloaded like any workspace file).

Why this exists:
  - Your Workspaces UI may not offer a direct download browser for @~ (user stage).
  - The notebook's /workspace filesystem *is* visible in the left nav.
  - So we bridge: stage → workspace file.

Outputs:
  - Workspace file created under: ./cpfr_exports/<backup_file>.csv
"""

from pathlib import Path
import os

# ---- 1) Identify the staged backup file we expect
STAGE_FOLDER = "cpfr_exports"
STAGE_FILE_NAME = f"{TARGET_TABLE}_backup_{RUN_TS}.csv"
STAGE_FILE = f"@~/{STAGE_FOLDER}/{STAGE_FILE_NAME}"

print("Expected staged file:", STAGE_FILE)

# ---- 2) Confirm it exists (LIST)
df_list = sf_query_df(f"LIST @~/{STAGE_FOLDER} PATTERN='.*{STAGE_FILE_NAME}.*'")
if df_list.empty:
    raise FileNotFoundError(
        f"Did not find {STAGE_FILE_NAME} in @~/{STAGE_FOLDER}. "
        "If RUN_TS changed, run sf_query_df('LIST @~/{STAGE_FOLDER}') and update STAGE_FILE_NAME."
    )

print("Found staged file:")
display(df_list)

# ---- 3) Read the staged file back as raw lines
# Snowflake returns stage-file columns as $1, $2...; $1 is the full line for our export
df_lines = sf_query_df(f"SELECT $1 AS line FROM {STAGE_FILE}")
print("Loaded staged lines:", df_lines.shape)
print("Columns returned:", list(df_lines.columns))
display(df_lines.head(5))

# Normalize column names to lower-case to avoid LINE vs line confusion
df_lines.columns = [c.lower() for c in df_lines.columns]
if "line" not in df_lines.columns:
    raise KeyError(f"Expected a 'line' column after normalization; got: {df_lines.columns.tolist()}")

# ---- 4) Write into the Workspaces filesystem (visible in left nav)
WORKSPACE_EXPORT_DIR = Path.cwd() / "cpfr_exports"
WORKSPACE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

workspace_path = WORKSPACE_EXPORT_DIR / STAGE_FILE_NAME

# Write WITHOUT header: each row is already a complete CSV line
df_lines["line"].to_csv(workspace_path, index=False, header=False)

print("Workspace copy written to:", workspace_path.resolve())
print("You should now see it in the left nav under: cpfr_exports/")
print("Files in cpfr_exports now:", os.listdir(WORKSPACE_EXPORT_DIR))

In [ ]:
import os
print(os.listdir(Path.cwd()))
print(os.listdir(Path.cwd() / "cpfr_exports"))

In [ ]:
# ==============================
# 8) BACKUP FOR DOWNLOAD VIA RESULTS GRID (No stages, no filesystem)
# ==============================
"""
Purpose:
  Produce a downloadable CSV via the Workspaces results grid.

How it works:
  - We return ONE column called LINE:
      - first row is the header line
      - subsequent rows are properly quoted CSV rows
  - Then you use the results grid "Download" / "Export" option (CSV).

Why this is reliable in Workspaces:
  - Doesn't depend on stage browsing or workspace filesystem visibility.
  - Works anywhere you can run SELECT and view results.

Download steps:
  - Run this cell
  - In the results output grid, use the download/export option (often a down-arrow icon)
"""

# (A) Get ordered column names
cols = sf_query_df(f"""
SELECT COLUMN_NAME
FROM {TARGET_DB}.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = '{TARGET_SCHEMA}'
  AND TABLE_NAME = '{TARGET_TABLE}'
ORDER BY ORDINAL_POSITION
""")["COLUMN_NAME"].tolist()

assert cols, "No columns found for target table; check TARGET_* config."

# (B) Build header + row expression (quoted + escaping)
header_line = ",".join([f'"{c}"' for c in cols])

csv_row_expr = " || ',' || ".join([
    f"""'"' || REPLACE(COALESCE(TO_VARCHAR("{c}"), ''), '"', '""') || '"'"""
    for c in cols
])

# (C) Return as query results (download from results grid)
df_download = sf_query_df(f"""
SELECT '{header_line}' AS "LINE"
UNION ALL
SELECT {csv_row_expr} AS "LINE"
FROM {TARGET_FQN}
""")

print("Lines returned (should be rowcount+1):", df_download.shape[0])
display(df_download.head(10))

In [ ]:
df_full = sf_query_df(f"SELECT * FROM {TARGET_FQN}")
display(df_full)